# 3-ML Pipeline approach

In [54]:
import os

# point java home to actual conda package reference
os.environ["JAVA_HOME"] = "/Users/andreasliistro/mambaforge/pkgs/openjdk-22.0.1-hbeb2e11_0/lib/jvm"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, MinMaxScaler
from pyspark.sql.functions import isnan, when, count, col, lit
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder


# init spark session
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "4g").getOrCreate()

In [55]:
# read data
data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles_cleaned.csv", header=True, inferSchema=True, multiLine=True)
data.show()

+----------+-------+------+------------+--------------------+---------+------+--------+------------+
|        id|  price|  year|manufacturer|               model|condition|  fuel|odometer|transmission|
+----------+-------+------+------------+--------------------+---------+------+--------+------------+
|7315799907| 5480.0|2013.0|   chevrolet|       captiva sport|     NULL|   gas|182711.0|   automatic|
|7315432897| 4100.0|2010.0|     hyundai|     elantra touring|     NULL|   gas|130206.0|      manual|
|7314812134|34500.0|2012.0|         bmw|                  x6|     NULL|   gas| 53723.0|   automatic|
|7314798728| 3200.0|1988.0|   chevrolet|          s10 pickup|     NULL|   gas|191000.0|   automatic|
|7312124399| 4100.0|2010.0|     hyundai|     elantra touring|     NULL|   gas|130206.0|      manual|
|7309034279| 4100.0|2010.0|     hyundai|     elantra touring|     NULL|   gas|130206.0|      manual|
|7308773221| 5480.0|2013.0|   chevrolet|       captiva sport|     NULL|   gas|182711.0|   a

In [56]:
data.printSchema()

root
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- year: double (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- transmission: string (nullable = true)



## Data Cleaning: Handle missing values

In [57]:
# show nulls
def show_nulls(dataframe):
  dataframe.select([count(when(col(c).isNull(), c)).alias(c) for c in dataframe.columns]).show()

show_nulls(data)

+---+-----+----+------------+-----+---------+----+--------+------------+
| id|price|year|manufacturer|model|condition|fuel|odometer|transmission|
+---+-----+----+------------+-----+---------+----+--------+------------+
|  0|    0|1122|       12984| 5286|   166151|2335|    4139|        2475|
+---+-----+----+------------+-----+---------+----+--------+------------+



In [58]:
# drop nulls for critical columns

# new_data = data.filter(~(col("manufacturer").isNull() & col("model").isNull())) # check where manufacturer & model are null and drop them
data = data.filter(~(col("manufacturer").isNull() | col("model").isNull())) # check where manufacturer or model are null and drop them
data = data.filter(~(col("year").isNull()))
show_nulls(data)

# replace nulls with unknown for condition, fuel and transmission
data = data.fillna("unknown", subset=["condition", "fuel", "transmission"])
show_nulls(data)

# replace nulls with mean for odometer (use groupby year to get mean)
data = data.fillna(data.groupBy("year").avg("odometer").first()[0], subset=["odometer"])
show_nulls(data)

+---+-----+----+------------+-----+---------+----+--------+------------+
| id|price|year|manufacturer|model|condition|fuel|odometer|transmission|
+---+-----+----+------------+-----+---------+----+--------+------------+
|  0|    0|   0|           0|    0|   157584|1918|    3917|        2232|
+---+-----+----+------------+-----+---------+----+--------+------------+

+---+-----+----+------------+-----+---------+----+--------+------------+
| id|price|year|manufacturer|model|condition|fuel|odometer|transmission|
+---+-----+----+------------+-----+---------+----+--------+------------+
|  0|    0|   0|           0|    0|        0|   0|    3917|           0|
+---+-----+----+------------+-----+---------+----+--------+------------+

+---+-----+----+------------+-----+---------+----+--------+------------+
| id|price|year|manufacturer|model|condition|fuel|odometer|transmission|
+---+-----+----+------------+-----+---------+----+--------+------------+
|  0|    0|   0|           0|    0|        0|   0

## Data Cleaning: Handle obvious duplicates

Remove duplicated entries based on year, manufacturer, model, odometer and price as this combination is highly unlikely to be duplicated.

In [59]:
# remove obvious duplicates (based on year, manufacturer, model, odometer, price)
data = data.dropDuplicates(subset=["year", "manufacturer", "model", "odometer", "price"])


## Data Processing: Scale numeric values

In [60]:
temp_data = data

# apply standard scaler to odometer
assembler = VectorAssembler(inputCols=["odometer"], outputCol="odometer_vec")
temp_data = assembler.transform(data)

scaler = StandardScaler(inputCol="odometer_vec", outputCol="odometer_scaled")
scaler_model = scaler.fit(temp_data)
temp_data = scaler_model.transform(temp_data)

# apply standard scaler to price
assembler = VectorAssembler(inputCols=["price"], outputCol="price_vec")
temp_data = assembler.transform(temp_data)

scaler = StandardScaler(inputCol="price_vec", outputCol="price_scaled")
scaler_model = scaler.fit(temp_data)
temp_data = scaler_model.transform(temp_data)

# apply standard scaler to year
assembler = VectorAssembler(inputCols=["year"], outputCol="year_vec")
temp_data = assembler.transform(temp_data)

scaler = MinMaxScaler(inputCol="year_vec", outputCol="year_scaled")
scaler_model = scaler.fit(temp_data)
temp_data = scaler_model.transform(temp_data)

# move scaled columns to new data
temp_data = temp_data.select("id", "price_scaled", "year_scaled", "odometer_scaled")
data = data.join(temp_data, on="id", how="inner")

data.show()

+----------+-------+------+-------------+--------------------+---------+-----+--------+------------+--------------------+--------------------+--------------------+
|        id|  price|  year| manufacturer|               model|condition| fuel|odometer|transmission|        price_scaled|         year_scaled|     odometer_scaled|
+----------+-------+------+-------------+--------------------+---------+-----+--------+------------+--------------------+--------------------+--------------------+
|7301589118|25000.0|2017.0|mercedes-benz|                benz|excellent|  gas| 51000.0|   automatic|[0.00162189037879...| [0.959016393442623]|[0.2464094953450586]|
|7301589688| 4500.0|2000.0|      mercury|       grand marquis|excellent|  gas|140000.0|   automatic|[2.91940268183104...| [0.819672131147541]| [0.676418222515847]|
|7301594281|43990.0|2020.0|        buick|enclave avenir sport|     good|  gas|  5217.0|       other|[0.00285387831052...|[0.9836065573770493]|[0.02520624190617...|
|7301595551|2799

## Data Processing: Encoding

Apply appropriate encoding

In [64]:
# apply one hot encoding to condition, fuel, transmission and manufacturer
condition_indexer = StringIndexer(inputCol="condition", outputCol="condition_index")
condition_encoder = OneHotEncoder(inputCol="condition_index", outputCol="condition_encoded")

fuel_indexer = StringIndexer(inputCol="fuel", outputCol="fuel_index")
fuel_encoder = OneHotEncoder(inputCol="fuel_index", outputCol="fuel_encoded")

transmission_indexer = StringIndexer(inputCol="transmission", outputCol="transmission_index")
transmission_encoder = OneHotEncoder(inputCol="transmission_index", outputCol="transmission_encoded")

manufacturer_indexer = StringIndexer(inputCol="manufacturer", outputCol="manufacturer_index")
manufacturer_encoder = OneHotEncoder(inputCol="manufacturer_index", outputCol="manufacturer_encoded")

# assembler = VectorAssembler(inputCols=["condition_encoded", "fuel_encoded", "transmission_encoded", "manufacturer_encoded", "year_scaled", "odometer_scaled"], outputCol="features")

pipeline = Pipeline(stages=[condition_indexer, condition_encoder, fuel_indexer, fuel_encoder, transmission_indexer, transmission_encoder, manufacturer_indexer, manufacturer_encoder])

pipeline_model = pipeline.fit(data)
new_data = pipeline_model.transform(data)

new_data = new_data.select("id", "condition_encoded", "fuel_encoded", "transmission_encoded", "manufacturer_encoded")
data = data.join(new_data, on="id", how="inner")

data.show()

+----------+-------+------+-------------+--------------------+---------+-----+--------+------------+--------------------+--------------------+--------------------+-----------------+-------------+--------------------+--------------------+
|        id|  price|  year| manufacturer|               model|condition| fuel|odometer|transmission|        price_scaled|         year_scaled|     odometer_scaled|condition_encoded| fuel_encoded|transmission_encoded|manufacturer_encoded|
+----------+-------+------+-------------+--------------------+---------+-----+--------+------------+--------------------+--------------------+--------------------+-----------------+-------------+--------------------+--------------------+
|7301589118|25000.0|2017.0|mercedes-benz|                benz|excellent|  gas| 51000.0|   automatic|[0.00162189037879...| [0.959016393442623]|[0.2464094953450586]|    (6,[1],[1.0])|(5,[0],[1.0])|       (3,[0],[1.0])|     (40,[12],[1.0])|
|7301589688| 4500.0|2000.0|      mercury|       

## ML-Pipeline

Create ML pipeline on existing data

In [71]:
# create a vector assembler to combine all features
assembler = VectorAssembler(inputCols=["condition_encoded", "fuel_encoded", "transmission_encoded", "manufacturer_encoded", "year_scaled", "odometer_scaled"], outputCol="features")

#create a regressor to predict car price
regressor = RandomForestRegressor(featuresCol='features', labelCol='price')

pipeline = Pipeline(stages=[assembler, regressor])

#save pipeline
pipeline.write().overwrite().save('pipeline')

In [72]:
#load pipeline
pipelineModel = Pipeline.load('pipeline')

#build paramgrid
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [1, 500]).build()

#build crossvalidator
crossval = CrossValidator(estimator=pipelineModel,
                          estimatorParamMaps=paramGrid,
                          evaluator=RegressionEvaluator(labelCol='price'), #price is the column we want to predict
                          numFolds=10)

In [73]:
#train test split
temp_data, test_data = data.randomSplit([0.9, 0.1], seed=123)
train_data, val_data = temp_data.randomSplit([0.9, 0.1], seed=123)

In [74]:
#fit
cvModel = crossval.fit(train_data)

#extract best model and view all the stages of the pipeline that our data went through
bestModel = cvModel.bestModel
for x in range(len(bestModel.stages)):
  print(bestModel.stages[x])

25/01/07 14:22:27 WARN CacheManager: Asked to cache already cached data.
25/01/07 14:22:27 WARN CacheManager: Asked to cache already cached data.
25/01/07 14:22:43 WARN DAGScheduler: Broadcasting large task binary with size 1056.5 KiB
25/01/07 14:22:48 WARN DAGScheduler: Broadcasting large task binary with size 1829.8 KiB
25/01/07 14:22:55 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
25/01/07 14:23:20 WARN DAGScheduler: Broadcasting large task binary with size 1057.1 KiB
25/01/07 14:23:25 WARN DAGScheduler: Broadcasting large task binary with size 1832.4 KiB
25/01/07 14:23:31 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
25/01/07 14:23:52 WARN DAGScheduler: Broadcasting large task binary with size 1056.5 KiB
25/01/07 14:23:57 WARN DAGScheduler: Broadcasting large task binary with size 1832.7 KiB
25/01/07 14:24:04 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
25/01/07 14:24:26 WARN DAGScheduler: Broadcasting large task b

VectorAssembler_ac0c92b44669
RandomForestRegressionModel: uid=RandomForestRegressor_8215b5f8cd94, numTrees=500, numFeatures=56


## Evaluate model

In [75]:
#transform the test set (use cvModel as it knows to pick the best model to use)
pred = cvModel.transform(test_data)
pred.select('price', 'prediction').show()

+-------+------------------+
|  price|        prediction|
+-------+------------------+
|43990.0| 25771.37141212526|
|39590.0|38161.087035388715|
|33590.0| 22804.31733885965|
|75000.0|166991.85595208572|
|  199.0|20029.055943177627|
|24990.0| 21384.06077751432|
|32990.0|18077.507578561817|
|58995.0|41578.367601216065|
|34999.0|20662.761178107772|
|12000.0| 161932.3418990113|
| 4495.0|20758.509213506462|
|89999.0| 41683.09317697143|
| 6500.0|148276.81644650703|
|25995.0|26305.995225967323|
|18980.0| 78031.60079775883|
| 8950.0| 17843.80367724527|
|19999.0| 20151.76469888939|
| 1500.0|409860.79748630885|
| 4495.0| 13283.96503959829|
|38000.0|21801.962203910578|
+-------+------------------+
only showing top 20 rows



In [76]:
#evaluate
eval = RegressionEvaluator(labelCol='price')

#get rmse
rmse = eval.evaluate(pred)

#get mse
mse = eval.evaluate(pred, {eval.metricName:'mse'})

#get mae
mae = eval.evaluate(pred, {eval.metricName:'mae'})

#get r2
r2 = eval.evaluate(pred, {eval.metricName:'r2'})

#print
print('RMSE: %3f' %rmse)
print('MSE: %3f' %mse)
print('MAE: %3f' %mae)
print('R2: %3f' %r2)

RMSE: 1354281.806818
MSE: 1834079212277.623047
MAE: 103291.284316
R2: -5183.148374
